# BioRob Phase 1A — Final Organized Cleaning + Audit Notebook

This notebook replaces the single long Phase 1A code cell with a clean, cell-by-cell workflow.

**Main fixes included**
1. Uses the correct project path: `/home/tsultan1/BioRob/Human Subject Data`
2. Parses filenames like `067_T116.csv` as `task=1`, `trial=16`
3. Correctly removes slide/media metadata rows such as `StartSlide`, `StartMedia`, `EndMedia`, and `EndSlide`
4. Drops all `EventSource`, `EventSource.1`, `EventSource.2`, `EventSource.3`, etc.
5. Saves per-file and per-subject audit reports
6. Prints `✅ CLEANING APPLIED CORRECTLY` only when required checks pass

In [1]:
# ============================================================
# CELL 1 — Imports and display settings
# This cell loads all packages used for Phase 1A cleaning.
# ============================================================

from pathlib import Path
import re
import json
import warnings
from typing import Optional, Dict, List, Tuple

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

print("✅ Imports loaded successfully.")

✅ Imports loaded successfully.


In [2]:
# ============================================================
# CELL 2 — Project configuration
# This cell defines the BioRob root path and rerun settings.
# ============================================================

ROOT_DIR = Path("/home/tsultan1/BioRob/Human Subject Data")
SUBJECT_GLOB = "Sub-*"

# Final rerun setting:
# True  = overwrite existing cleaned CSVs
# False = skip if cleaned CSV already exists
OVERWRITE_CLEANED = True

# Print one line verdict for every file.
VERBOSE_EACH_FILE = True

# Output reports are saved inside the BioRob project folder.
REPORT_DIR = ROOT_DIR / "_audit_phase1a_cleaning"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Expected Neon world-camera field of view used by your original Phase 1A code.
ET_FOV_W = 1600.0
ET_FOV_H = 1200.0

# Keep distances and pupil sizes in millimeters, same as your original code.
KEEP_DIST_IN_MM = True

# Keep raw EMG and drop vendor mV EMG, same as your original code.
KEEP_EMG_RAW = True
DROP_EMG_VENDOR = True

# Drop eyelid features unless you plan to use them downstream.
DROP_EYELID_FEATURES = True

print("ROOT_DIR:", ROOT_DIR)
print("REPORT_DIR:", REPORT_DIR)
print("OVERWRITE_CLEANED:", OVERWRITE_CLEANED)

ROOT_DIR: /home/tsultan1/BioRob/Human Subject Data
REPORT_DIR: /home/tsultan1/BioRob/Human Subject Data/_audit_phase1a_cleaning
OVERWRITE_CLEANED: True


In [3]:
# ============================================================
# CELL 3 — Check project folder and list subjects
# This cell confirms the root folder exists and prints all detected subjects.
# ============================================================

if not ROOT_DIR.exists():
    raise FileNotFoundError(f"ROOT_DIR does not exist: {ROOT_DIR}")

subj_dirs = sorted(
    [p for p in ROOT_DIR.glob(SUBJECT_GLOB) if p.is_dir()],
    key=lambda p: int(re.search(r"Sub[-_ ]*(\d+)", p.name, re.I).group(1)) if re.search(r"Sub[-_ ]*(\d+)", p.name, re.I) else 10**9
)

if not subj_dirs:
    raise RuntimeError(f"No subject folders found under: {ROOT_DIR}")

subject_ids = []
for p in subj_dirs:
    m = re.search(r"Sub[-_ ]*(\d+)", p.name, re.I)
    if m:
        subject_ids.append(int(m.group(1)))

print(f"✅ Found {len(subj_dirs)} subject folders.")
print("Subjects:", [f"Sub-{sid}" for sid in subject_ids])

raw_counts = []
for p in subj_dirs:
    csvs = sorted([x for x in p.glob("*.csv") if x.is_file()])
    raw_counts.append({"subject": p.name, "raw_csv_count": len(csvs)})

raw_count_df = pd.DataFrame(raw_counts)
display(raw_count_df)
print("Total raw CSV files:", int(raw_count_df["raw_csv_count"].sum()))

✅ Found 20 subject folders.
Subjects: ['Sub-1', 'Sub-2', 'Sub-3', 'Sub-5', 'Sub-6', 'Sub-7', 'Sub-8', 'Sub-9', 'Sub-10', 'Sub-11', 'Sub-12', 'Sub-13', 'Sub-14', 'Sub-16', 'Sub-17', 'Sub-19', 'Sub-20', 'Sub-21', 'Sub-22', 'Sub-23']


,subject,raw_csv_count
0,Sub-1,83
1,Sub-2,83
2,Sub-3,83
3,Sub-5,83
4,Sub-6,83
5,Sub-7,83
6,Sub-8,82
7,Sub-9,83
8,Sub-10,83
9,Sub-11,82


Total raw CSV files: 1657


In [4]:
# ============================================================
# CELL 4 — Constants and required schema checks
# This cell defines expected EEG, EMG, and eye-tracking columns.
# ============================================================

EEG_COLS = [f"Ch{i}" for i in range(1, 9)]

EMG_RAW_COLS = ["Ch1 EMG raw", "Ch2 EMG raw", "Ch3 EMG raw", "Ch4 EMG raw"]
EMG_VENDOR_COLS = ["Ch1 EMG", "Ch2 EMG", "Ch3 EMG", "Ch4 EMG"]

ET_GAZE_2D_COLS = ["ET_GazeLeftx", "ET_GazeLefty", "ET_GazeRightx", "ET_GazeRighty"]
ET_DIST_PUPIL_COLS = ["ET_DistanceLeft", "ET_DistanceRight", "ET_PupilLeft", "ET_PupilRight"]
ET_FLAG_COLS = ["ET_ValidityLeftEye", "ET_ValidityRightEye", "ET_Blink", "ET_Fixation", "ET_Worn"]
ET_IMU_HEAD_COLS = [
    "ET_GyroX", "ET_GyroY", "ET_GyroZ",
    "ET_AccX", "ET_AccY", "ET_AccZ",
    "ET_HeadRotationPitch", "ET_HeadRotationYaw", "ET_HeadRotationRoll",
]
ET_3D_COLS = [
    "ET_Gaze3dEyeballXLeft", "ET_Gaze3dEyeballYLeft", "ET_Gaze3dEyeballZLeft",
    "ET_Gaze3dEyeballXRight", "ET_Gaze3dEyeballYRight", "ET_Gaze3dEyeballZRight",
    "ET_Gaze3dOpticalAxisXLeft", "ET_Gaze3dOpticalAxisYLeft", "ET_Gaze3dOpticalAxisZLeft",
    "ET_Gaze3dOpticalAxisXRight", "ET_Gaze3dOpticalAxisYRight", "ET_Gaze3dOpticalAxisZRight",
    "ET_Gaze3dEyelidAngleTopLeft", "ET_Gaze3dEyelidAngleBottomLeft",
    "ET_Gaze3dEyelidAngleTopRight", "ET_Gaze3dEyelidAngleBottomRight",
    "ET_Gaze3dEyelidApertureLeft", "ET_Gaze3dEyelidApertureRight",
]

ET_EYELID_COLS = [
    "ET_Gaze3dEyelidAngleTopLeft", "ET_Gaze3dEyelidAngleBottomLeft",
    "ET_Gaze3dEyelidAngleTopRight", "ET_Gaze3dEyelidAngleBottomRight",
    "ET_Gaze3dEyelidApertureLeft", "ET_Gaze3dEyelidApertureRight",
]

# Columns that should not remain in cleaned training files.
DROP_COLS_EXACT_BASE = {
    "EventSource", "SlideEvent", "StimType", "Duration",
    "CollectionPhase", "SourceStimuliName",
    "Auto 1 active", "Auto 1 instance",
    "Active active", "active active", "active instance",
    "ET_CameraLeftX", "ET_CameraLeftY", "ET_CameraRightX", "ET_CameraRightY",
}

# Required columns after cleaning.
# If one of these is missing, the file is not considered clean.
REQUIRED_AFTER_CLEAN = (
    ["subject_id", "task", "trial", "Timestamp_seconds"]
    + EEG_COLS
    + EMG_RAW_COLS
    + [
        "ET_TimeSignal",
        "ET_PupilLeft", "ET_PupilRight",
        "ET_DistanceLeft", "ET_DistanceRight",
        "ET_GazeLeftx", "ET_GazeLefty",
        "ET_GazeRightx", "ET_GazeRighty",
        "ET_ValidityLeftEye", "ET_ValidityRightEye",
        "ET_Blink", "ET_Fixation", "ET_Worn",
    ]
)

print("✅ Schema constants are ready.")
print("Required column count:", len(REQUIRED_AFTER_CLEAN))

✅ Schema constants are ready.
Required column count: 30


In [5]:
# ============================================================
# CELL 5 — Filename and header helpers
# This cell fixes the important T116 parsing issue.
# Example: 067_T116.csv -> task=1, trial=16
# ============================================================

TRIAL_RE = re.compile(r"[_\-](?P<kind>[TM])(?P<num>\d+)\.csv$", re.IGNORECASE)

def parse_subject_id(folder_name: str) -> Optional[int]:
    m = re.search(r"Sub[\s\-_]*(\d+)", folder_name, re.IGNORECASE)
    return int(m.group(1)) if m else None

def parse_task_trial(filename: str) -> Tuple[Optional[int], Optional[int]]:
    """
    BioRob convention used here:
      T116 -> task 1, trial 16
      T26  -> task 2, trial 6
      T05  -> task 0, trial 5
    This avoids the old mistake where T116 became task 11, trial 6.
    """
    m = TRIAL_RE.search(filename)
    if not m:
        return None, None

    digits = m.group("num")
    if len(digits) < 2:
        return None, None

    task = int(digits[0])
    trial = int(digits[1:])
    return task, trial

def find_header_row_index(path: Path) -> int:
    """
    Finds the real CSV header row that starts with 'Row,'.
    iMotions/Neon exports often contain metadata lines before the real data table.
    """
    with path.open("r", encoding="utf-8-sig", errors="ignore") as f:
        for i, line in enumerate(f):
            if line.lstrip().startswith("Row,"):
                return i
    raise ValueError(f"Could not find header row starting with 'Row,' in {path}")

def read_raw_imotions_csv(path: Path) -> pd.DataFrame:
    header_idx = find_header_row_index(path)
    df = pd.read_csv(
        path,
        skiprows=header_idx,
        header=0,
        encoding="utf-8-sig",
        engine="python",
        on_bad_lines="skip",
    )
    df.columns = [str(c).strip() for c in df.columns]
    return df

# Test the parser before touching any data.
for ex in ["067_T116.csv", "061_T26.csv", "001_T05.csv", "062_T106.csv"]:
    print(ex, "->", parse_task_trial(ex))

print("✅ Filename parser ready.")

067_T116.csv -> (1, 16)
061_T26.csv -> (2, 6)
001_T05.csv -> (0, 5)
062_T106.csv -> (1, 6)
✅ Filename parser ready.


In [6]:
# ============================================================
# CELL 6 — Cleaning functions
# This cell applies your original Phase 1A cleaning logic with bug fixes.
# ============================================================

def _coerce_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")

def drop_slide_metadata_rows(df: pd.DataFrame) -> Tuple[pd.DataFrame, int]:
    """
    Removes pure metadata/media rows such as StartSlide, StartMedia, EndMedia, EndSlide.

    Important fix:
    The old code checked EventSource != 'SlideEvents', but in your sample file
    EventSource is numeric and SlideEvent contains the actual event names.
    """
    df = df.copy()

    event_mask = pd.Series(False, index=df.index)

    for col in ["SlideEvent", "StimType", "CollectionPhase"]:
        if col in df.columns:
            event_mask |= df[col].notna()

    if "EventSource" in df.columns:
        event_source_str = df["EventSource"].astype(str)
        event_mask |= event_source_str.str.contains("SlideEvents|StartSlide|StartMedia|EndMedia|EndSlide", case=False, na=False)

    # Do not use Timestamp alone as a signal indicator.
    signal_indicator_cols = (
        ["SampleNumber", "ET_TimeSignal", "LSL Timestamp"]
        + EEG_COLS
        + EMG_RAW_COLS
        + ET_GAZE_2D_COLS
        + ET_DIST_PUPIL_COLS
    )
    present_signal_cols = [c for c in signal_indicator_cols if c in df.columns]

    if present_signal_cols:
        signal_present = df[present_signal_cols].notna().any(axis=1)
        drop_mask = event_mask & (~signal_present)
    else:
        drop_mask = event_mask

    removed = int(drop_mask.sum())
    df = df.loc[~drop_mask].copy()
    return df, removed

def make_timestamp_seconds(df: pd.DataFrame) -> Tuple[Optional[pd.Series], str, Dict[str, float]]:
    """
    Builds a master seconds clock while keeping original timing columns.
    Priority follows your original code:
      Timestamp ms -> ET_TimeSignal ms -> LSL Timestamp seconds
    """
    candidates = []
    if "Timestamp" in df.columns:
        candidates.append(("Timestamp", 1000.0))
    if "ET_TimeSignal" in df.columns:
        candidates.append(("ET_TimeSignal", 1000.0))
    if "LSL Timestamp" in df.columns:
        candidates.append(("LSL Timestamp", 1.0))

    for col, denom in candidates:
        s = pd.to_numeric(df[col], errors="coerce")
        if s.notna().any():
            t = s / denom
            first_valid = t.dropna().iloc[0]
            t = (t - first_valid).ffill().bfill()
            dt = t.diff().dropna()
            positive_dt = dt[dt > 0]
            diag = {
                "timestamp_valid_count": int(t.notna().sum()),
                "timestamp_min_s": float(t.min()) if t.notna().any() else np.nan,
                "timestamp_max_s": float(t.max()) if t.notna().any() else np.nan,
                "timestamp_positive_median_dt_s": float(positive_dt.median()) if len(positive_dt) else np.nan,
            }
            return t.astype("float64"), col, diag

    return None, "NONE", {}

def center_emg_raw_inplace(df: pd.DataFrame) -> List[str]:
    """
    Robust-centers raw EMG ADC-like channels by subtracting the channel median.
    This keeps your paper pipeline's raw-EMG policy but removes the ADC baseline.
    """
    logs = []
    for col in EMG_RAW_COLS:
        if col not in df.columns:
            continue
        s = _coerce_numeric(df[col])
        if not s.notna().any():
            df[col] = s.astype("float32")
            logs.append(f"{col}: all NaN")
            continue

        med = float(s.median())
        if 1e4 <= med <= 6e4:
            df[col] = (s - med).astype("float32")
            logs.append(f"{col}: centered by median {med:.2f}")
        else:
            df[col] = s.astype("float32")
            logs.append(f"{col}: not centered; median {med:.2f} not ADC-like")
    return logs

def apply_unit_conversions_inplace(df: pd.DataFrame) -> List[str]:
    """
    Applies unit conversions and dtype tightening:
      - gaze pixel coordinates -> [0,1]
      - pupil/distance kept in mm
      - ET flags -> uint8
      - EEG/EMG/ET numeric channels -> float32
    """
    logs = []

    gaze_denoms = {
        "ET_GazeLeftx": ET_FOV_W,
        "ET_GazeRightx": ET_FOV_W,
        "ET_GazeLefty": ET_FOV_H,
        "ET_GazeRighty": ET_FOV_H,
    }
    for col, denom in gaze_denoms.items():
        if col in df.columns:
            s = _coerce_numeric(df[col])
            df[col] = (s / denom).clip(lower=0.0, upper=1.0).astype("float32")
            logs.append(f"{col}: px -> [0,1]")

    for col in ET_DIST_PUPIL_COLS:
        if col in df.columns:
            s = _coerce_numeric(df[col])
            if KEEP_DIST_IN_MM:
                df[col] = s.astype("float32")
                logs.append(f"{col}: kept as mm")
            else:
                s2 = s.astype("float64")
                mask = s2 >= 0
                s2.loc[mask] = s2.loc[mask] / 1000.0
                df[col] = s2.astype("float32")
                logs.append(f"{col}: mm -> m")

    for col in ET_FLAG_COLS:
        if col in df.columns:
            s = _coerce_numeric(df[col])
            df[col] = (s.fillna(0) > 0.5).astype("uint8")
            logs.append(f"{col}: thresholded to uint8")

    for col in ET_IMU_HEAD_COLS + ET_3D_COLS:
        if col in df.columns:
            df[col] = _coerce_numeric(df[col]).astype("float32")

    for col in EEG_COLS:
        if col in df.columns:
            df[col] = _coerce_numeric(df[col]).astype("float32")

    logs += center_emg_raw_inplace(df)
    return logs

def drop_unneeded_columns(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drops metadata/noisy columns and all EventSource variants.
    Important fix: drops EventSource.3 too, not only EventSource.1 and EventSource.2.
    """
    to_drop = set(DROP_COLS_EXACT_BASE)

    if not KEEP_EMG_RAW:
        to_drop.update(EMG_RAW_COLS)
    if DROP_EMG_VENDOR:
        to_drop.update(EMG_VENDOR_COLS)
    if DROP_EYELID_FEATURES:
        to_drop.update(ET_EYELID_COLS)

    for c in df.columns:
        if re.match(r"^EventSource(\.\d+)?$", str(c)):
            to_drop.add(c)
        if str(c).startswith("Unnamed:"):
            to_drop.add(c)

    drop_now = sorted([c for c in to_drop if c in df.columns])
    df = df.drop(columns=drop_now, errors="ignore").copy()
    return df, drop_now

def compact_id_columns(df: pd.DataFrame) -> None:
    for col in ["subject_id", "task", "trial", "SampleNumber", "Row"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int32")

print("✅ Cleaning functions ready.")

✅ Cleaning functions ready.


In [7]:
# ============================================================
# CELL 7 — Validation function
# This cell checks if cleaning was applied correctly.
# ============================================================

def validate_cleaned_df(df: pd.DataFrame, filename: str) -> Tuple[bool, List[str], List[str], Dict[str, object]]:
    errors = []
    warnings_list = []
    metrics = {}

    metrics["rows_after"] = int(len(df))
    metrics["cols_after"] = int(len(df.columns))

    # Required columns
    missing_required = [c for c in REQUIRED_AFTER_CLEAN if c not in df.columns]
    metrics["missing_required_count"] = len(missing_required)
    metrics["missing_required"] = ";".join(missing_required[:20])
    if missing_required:
        errors.append(f"Missing required columns: {missing_required}")

    # Metadata columns should be gone
    leftover_event_cols = [c for c in df.columns if re.match(r"^EventSource(\.\d+)?$", str(c))]
    leftover_drop_cols = [c for c in df.columns if c in DROP_COLS_EXACT_BASE]
    leftover_bad_cols = leftover_event_cols + leftover_drop_cols
    metrics["leftover_bad_cols"] = ";".join(leftover_bad_cols)
    if leftover_bad_cols:
        errors.append(f"Metadata columns still present: {leftover_bad_cols}")

    # Task/trial sanity
    if "task" in df.columns:
        tasks = sorted(pd.Series(df["task"]).dropna().unique().tolist())
        metrics["tasks"] = str(tasks)
        if any((t < 0 or t > 5) for t in tasks):
            errors.append(f"Task id outside expected 0-5 range: {tasks}")

    if "trial" in df.columns:
        trials = sorted(pd.Series(df["trial"]).dropna().unique().tolist())
        metrics["trials"] = str(trials)

    # Timestamp sanity
    if "Timestamp_seconds" not in df.columns:
        errors.append("Timestamp_seconds missing")
    else:
        t = pd.to_numeric(df["Timestamp_seconds"], errors="coerce")
        metrics["timestamp_nan_count"] = int(t.isna().sum())
        metrics["duration_s"] = float(t.max() - t.min()) if t.notna().any() else np.nan

        dt = t.diff().dropna()
        neg_dt_count = int((dt < -1e-9).sum())
        metrics["timestamp_negative_dt_count"] = neg_dt_count
        if neg_dt_count > 0:
            errors.append(f"Timestamp_seconds is not monotonic; negative dt count={neg_dt_count}")

    # Gaze normalization check
    for col in ET_GAZE_2D_COLS:
        if col in df.columns and df[col].notna().any():
            mn = float(pd.to_numeric(df[col], errors="coerce").min())
            mx = float(pd.to_numeric(df[col], errors="coerce").max())
            metrics[f"{col}_min"] = mn
            metrics[f"{col}_max"] = mx
            if mn < -1e-6 or mx > 1.000001:
                errors.append(f"{col} not normalized to [0,1]: min={mn}, max={mx}")

    # EMG centering check
    for col in EMG_RAW_COLS:
        if col in df.columns and df[col].notna().any():
            med = float(pd.to_numeric(df[col], errors="coerce").median())
            metrics[f"{col}_median_after"] = med
            if abs(med) > 1e-3:
                warnings_list.append(f"{col} median after cleaning is {med:.3f}; check if it was not ADC-like or had unusual values")

    # EEG/EMG/ET coverage warnings
    for group_name, cols in {
        "EEG": EEG_COLS,
        "EMG": EMG_RAW_COLS,
        "ET": ["ET_TimeSignal"] + ET_GAZE_2D_COLS + ET_FLAG_COLS,
    }.items():
        present = [c for c in cols if c in df.columns]
        if present:
            coverage = float(df[present].notna().any(axis=1).mean())
            metrics[f"{group_name}_row_coverage"] = coverage
            if coverage < 0.01:
                warnings_list.append(f"{group_name} coverage is very low: {coverage:.4f}")

    ok = len(errors) == 0
    return ok, errors, warnings_list, metrics

print("✅ Validation function ready.")

✅ Validation function ready.


In [8]:
# ============================================================
# CELL 8 — Clean one file
# This cell combines reading, metadata insertion, cleaning, validation, and saving.
# ============================================================

def clean_one_file(csv_path: Path, save: bool = True) -> Dict[str, object]:
    subj_dir = csv_path.parent
    sid = parse_subject_id(subj_dir.name)
    task, trial = parse_task_trial(csv_path.name)

    if sid is None:
        raise ValueError(f"Could not parse subject id from folder: {subj_dir.name}")
    if task is None or trial is None:
        raise ValueError(f"Could not parse task/trial from filename: {csv_path.name}")

    raw_df = read_raw_imotions_csv(csv_path)
    rows_before = int(len(raw_df))
    cols_before = int(len(raw_df.columns))

    df = raw_df.copy()

    # Insert subject/task/trial at the front.
    for col, val in [("trial", trial), ("task", task), ("subject_id", sid)]:
        if col in df.columns:
            df[col] = val
        else:
            df.insert(0, col, val)

    # Remove pure slide/media rows before timestamp generation.
    df, slide_rows_removed = drop_slide_metadata_rows(df)

    # Add master timestamp immediately after subject/task/trial.
    if "Timestamp_seconds" not in df.columns:
        ts, ts_source, ts_diag = make_timestamp_seconds(df)
        if ts is not None:
            insert_at = 3
            df.insert(insert_at, "Timestamp_seconds", ts.values.astype("float64"))
        else:
            ts_source, ts_diag = "NONE", {}
    else:
        ts_source, ts_diag = "existing", {}

    # Unit conversion and dtype tightening.
    conversion_logs = apply_unit_conversions_inplace(df)

    # Drop unused columns.
    df, dropped_cols = drop_unneeded_columns(df)

    # Compact IDs.
    compact_id_columns(df)

    # Validate.
    ok, errors, warnings_list, metrics = validate_cleaned_df(df, csv_path.name)

    # Save output.
    out_dir = subj_dir / "cleaned"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / csv_path.name

    saved = False
    if save:
        if out_path.exists() and not OVERWRITE_CLEANED:
            saved = False
            warnings_list.append("Output already exists and OVERWRITE_CLEANED=False; file was not overwritten")
        else:
            df.to_csv(out_path, index=False, encoding="utf-8-sig")
            saved = True

    record = {
        "subject": subj_dir.name,
        "subject_id": sid,
        "file": csv_path.name,
        "input_path": str(csv_path),
        "output_path": str(out_path),
        "task": task,
        "trial": trial,
        "rows_before": rows_before,
        "cols_before": cols_before,
        "rows_after": int(len(df)),
        "cols_after": int(len(df.columns)),
        "slide_rows_removed": slide_rows_removed,
        "timestamp_source": ts_source,
        "dropped_cols_count": len(dropped_cols),
        "dropped_cols": ";".join(dropped_cols),
        "saved": bool(saved),
        "ok": bool(ok),
        "errors": " | ".join(errors),
        "warnings": " | ".join(warnings_list),
    }
    record.update(ts_diag)
    record.update(metrics)

    if VERBOSE_EACH_FILE:
        verdict = "✅ CLEANING APPLIED CORRECTLY" if ok else "❌ CLEANING CHECK FAILED"
        print(f"{verdict} | {subj_dir.name}/{csv_path.name} | task={task}, trial={trial} | rows {rows_before}->{len(df)} | saved={saved}")
        if errors:
            print("   Errors:", errors)
        if warnings_list:
            print("   Warnings:", warnings_list[:3])

    return record

print("✅ clean_one_file() ready.")

✅ clean_one_file() ready.


In [9]:
# ============================================================
# CELL 9 — Single-file dry-run check before full rerun
# This cell cleans one raw CSV without saving, so you can inspect the output.
# ============================================================

all_raw_files = []
for subj_dir in subj_dirs:
    all_raw_files.extend(sorted([p for p in subj_dir.glob("*.csv") if p.is_file()]))

if not all_raw_files:
    raise RuntimeError("No raw CSV files found directly inside Sub-* folders.")

# Prefer a T116 file if present, because this catches the old parsing mistake.
preferred = [p for p in all_raw_files if re.search(r"_T116\.csv$", p.name, re.I)]
sample_file = preferred[0] if preferred else all_raw_files[0]

print("Sample file selected:", sample_file)

sample_record = clean_one_file(sample_file, save=False)

print("\nSingle-file audit record:")
display(pd.DataFrame([sample_record]).T)

print("\n✅ Single-file dry-run finished. If the verdict above is correct, run the batch cell next.")

Sample file selected: /home/tsultan1/BioRob/Human Subject Data/Sub-1/034_T116.csv
✅ CLEANING APPLIED CORRECTLY | Sub-1/034_T116.csv | task=1, trial=16 | rows 19034->19030 | saved=False

Single-file audit record:


,0
subject,Sub-1
subject_id,1
file,034_T116.csv
input_path,/home/tsultan1/BioRob/Human Subject Data/Sub-1...
output_path,/home/tsultan1/BioRob/Human Subject Data/Sub-1...
task,1
trial,16
rows_before,19034
cols_before,74
rows_after,19030



✅ Single-file dry-run finished. If the verdict above is correct, run the batch cell next.


In [10]:
# ============================================================
# CELL 10 — Full batch cleaning for all subjects
# This cell reruns Phase 1A for every raw CSV in every Sub-* folder.
# ============================================================

audit_records = []
failed_files = []

print(f"Starting full Phase 1A cleaning on {len(all_raw_files)} raw files...")
print("OVERWRITE_CLEANED:", OVERWRITE_CLEANED)

for csv_path in all_raw_files:
    try:
        rec = clean_one_file(csv_path, save=True)
        audit_records.append(rec)
        if not rec["ok"]:
            failed_files.append(rec)
    except Exception as e:
        err_rec = {
            "subject": csv_path.parent.name,
            "file": csv_path.name,
            "input_path": str(csv_path),
            "ok": False,
            "saved": False,
            "errors": repr(e),
        }
        audit_records.append(err_rec)
        failed_files.append(err_rec)
        print(f"❌ ERROR | {csv_path.parent.name}/{csv_path.name}: {e}")

audit_df = pd.DataFrame(audit_records)
audit_path = REPORT_DIR / "phase1a_cleaning_audit.csv"
audit_df.to_csv(audit_path, index=False)

failed_df = pd.DataFrame(failed_files)
failed_path = REPORT_DIR / "phase1a_failed_files.csv"
failed_df.to_csv(failed_path, index=False)

print("\n============================================================")
print("PHASE 1A BATCH CLEANING SUMMARY")
print("============================================================")
print("Total files processed:", len(audit_df))
print("Files passed:", int(audit_df["ok"].fillna(False).sum()) if "ok" in audit_df.columns else 0)
print("Files failed:", len(failed_df))
print("Audit saved to:", audit_path)
print("Failed-file report saved to:", failed_path)

if len(failed_df) == 0:
    print("✅ ALL PHASE 1A FILES CLEANED CORRECTLY")
else:
    print("❌ SOME FILES FAILED. Displaying failed files:")
    cols = ["subject", "file", "errors"]
    if "warnings" in failed_df.columns:
        cols.append("warnings")
    display(failed_df[cols])

Starting full Phase 1A cleaning on 1657 raw files...
OVERWRITE_CLEANED: True
✅ CLEANING APPLIED CORRECTLY | Sub-1/001_T03.csv | task=0, trial=3 | rows 16791->16787 | saved=True
✅ CLEANING APPLIED CORRECTLY | Sub-1/002_T516.csv | task=5, trial=16 | rows 14139->14135 | saved=True
✅ CLEANING APPLIED CORRECTLY | Sub-1/003_T515.csv | task=5, trial=15 | rows 17460->17456 | saved=True
✅ CLEANING APPLIED CORRECTLY | Sub-1/004_T514.csv | task=5, trial=14 | rows 14955->14951 | saved=True
✅ CLEANING APPLIED CORRECTLY | Sub-1/005_T513.csv | task=5, trial=13 | rows 15423->15419 | saved=True
✅ CLEANING APPLIED CORRECTLY | Sub-1/006_T512.csv | task=5, trial=12 | rows 17492->17488 | saved=True
✅ CLEANING APPLIED CORRECTLY | Sub-1/007_T511.csv | task=5, trial=11 | rows 17113->17109 | saved=True
✅ CLEANING APPLIED CORRECTLY | Sub-1/008_T510.csv | task=5, trial=10 | rows 16579->16575 | saved=True
✅ CLEANING APPLIED CORRECTLY | Sub-1/009_T59.csv | task=5, trial=9 | rows 12619->12615 | saved=True
✅ CLEANIN

,subject,file,errors,warnings
0,Sub-3,071_T112.csv,Missing required columns: ['ET_ValidityLeftEye...,
1,Sub-6,036_T315.csv,Missing required columns: ['ET_ValidityLeftEye...,
2,Sub-6,074_T9.csv,ValueError('Could not parse task/trial from fi...,NaN
3,Sub-7,038_T313.csv,Missing required columns: ['ET_ValidityLeftEye...,
4,Sub-8,006_T512.csv,Missing required columns: ['ET_ValidityLeftEye...,
...,...,...,...,...
100,Sub-22,011_T57.csv,Missing required columns: ['ET_ValidityLeftEye...,
101,Sub-22,029_T45.csv,Missing required columns: ['ET_ValidityLeftEye...,
102,Sub-23,039_T312.csv,Missing required columns: ['ET_ValidityLeftEye...,
103,Sub-23,055_T212.csv,Missing required columns: ['ET_ValidityLeftEye...,


In [11]:
# ============================================================
# CELL 11 — Subject-level summary after cleaning
# This cell summarizes cleaned files per subject and important sanity metrics.
# ============================================================

if "audit_df" not in globals():
    audit_path = REPORT_DIR / "phase1a_cleaning_audit.csv"
    audit_df = pd.read_csv(audit_path)

summary = (
    audit_df
    .groupby(["subject", "subject_id"], dropna=False)
    .agg(
        files=("file", "count"),
        passed=("ok", lambda x: int(pd.Series(x).fillna(False).sum())),
        failed=("ok", lambda x: int((~pd.Series(x).fillna(False).astype(bool)).sum())),
        min_task=("task", "min"),
        max_task=("task", "max"),
        min_trial=("trial", "min"),
        max_trial=("trial", "max"),
        total_rows_after=("rows_after", "sum"),
    )
    .reset_index()
    .sort_values("subject_id")
)

summary_path = REPORT_DIR / "phase1a_subject_summary.csv"
summary.to_csv(summary_path, index=False)

print("Subject summary saved to:", summary_path)
display(summary)

if int(summary["failed"].sum()) == 0:
    print("✅ SUBJECT-LEVEL CHECK PASSED")
else:
    print("❌ SUBJECT-LEVEL CHECK FAILED")

Subject summary saved to: /home/tsultan1/BioRob/Human Subject Data/_audit_phase1a_cleaning/phase1a_subject_summary.csv


,subject,subject_id,files,passed,failed,min_task,max_task,min_trial,max_trial,total_rows_after
0,Sub-1,1.0,83,83,0,0.0,5.0,1.0,16.0,1433213.0
9,Sub-2,2.0,83,83,0,0.0,5.0,1.0,16.0,1821035.0
14,Sub-3,3.0,83,82,1,0.0,5.0,1.0,16.0,1566913.0
15,Sub-5,5.0,83,83,0,0.0,5.0,1.0,16.0,1167859.0
16,Sub-6,6.0,82,81,1,0.0,5.0,1.0,16.0,1107863.0
18,Sub-7,7.0,83,82,1,0.0,5.0,1.0,16.0,1240719.0
19,Sub-8,8.0,82,80,2,0.0,5.0,1.0,16.0,1162734.0
20,Sub-9,9.0,83,82,1,0.0,5.0,1.0,16.0,1110037.0
1,Sub-10,10.0,83,82,1,0.0,5.0,1.0,16.0,1180669.0
2,Sub-11,11.0,82,80,2,0.0,5.0,1.0,16.0,1027067.0


❌ SUBJECT-LEVEL CHECK FAILED


In [12]:
# ============================================================
# CELL 12 — Verify cleaned folders and cleaned CSV existence
# This cell physically checks that cleaned CSV files exist on disk.
# ============================================================

exist_records = []

for _, row in audit_df.iterrows():
    out_path = Path(row["output_path"]) if "output_path" in row and pd.notna(row["output_path"]) else None
    exists = out_path.exists() if out_path is not None else False
    exist_records.append({
        "subject": row.get("subject"),
        "file": row.get("file"),
        "output_path": str(out_path) if out_path is not None else "",
        "exists": exists,
        "ok": bool(row.get("ok", False)),
    })

exist_df = pd.DataFrame(exist_records)
missing_outputs = exist_df[(exist_df["ok"] == True) & (exist_df["exists"] == False)]

exist_path = REPORT_DIR / "phase1a_output_existence_check.csv"
exist_df.to_csv(exist_path, index=False)

print("Output existence report saved to:", exist_path)
print("Cleaned outputs missing for passed files:", len(missing_outputs))

if len(missing_outputs) == 0:
    print("✅ CLEANED CSV FILES EXIST ON DISK")
else:
    print("❌ Some passed files do not have saved outputs:")
    display(missing_outputs)

Output existence report saved to: /home/tsultan1/BioRob/Human Subject Data/_audit_phase1a_cleaning/phase1a_output_existence_check.csv
Cleaned outputs missing for passed files: 0
✅ CLEANED CSV FILES EXIST ON DISK


In [13]:
# ============================================================
# CELL 13 — Save final cleaned schema
# This cell saves the final column order from one cleaned file for reproducibility.
# ============================================================

passed = audit_df[audit_df["ok"] == True].copy()
if len(passed) == 0:
    raise RuntimeError("No passed cleaned files found; cannot save schema.")

first_out = Path(passed.iloc[0]["output_path"])
schema_df = pd.read_csv(first_out, nrows=5)
schema = {
    "example_file": str(first_out),
    "n_columns": len(schema_df.columns),
    "columns": list(schema_df.columns),
}

schema_path = REPORT_DIR / "phase1a_cleaned_schema.json"
with open(schema_path, "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2)

print("Schema saved to:", schema_path)
print("Number of columns:", schema["n_columns"])
print("First 30 columns:")
print(schema["columns"][:30])

print("\n✅ PHASE 1A FINAL CLEANING PIPELINE COMPLETE")

Schema saved to: /home/tsultan1/BioRob/Human Subject Data/_audit_phase1a_cleaning/phase1a_cleaned_schema.json
Number of columns: 55
First 30 columns:
['subject_id', 'task', 'trial', 'Timestamp_seconds', 'Row', 'Timestamp', 'SampleNumber', 'Ch1 EMG raw', 'Ch2 EMG raw', 'Ch3 EMG raw', 'Ch4 EMG raw', 'ET_TimeSignal', 'ET_PupilLeft', 'ET_PupilRight', 'ET_DistanceLeft', 'ET_DistanceRight', 'ET_GazeLeftx', 'ET_GazeLefty', 'ET_GazeRightx', 'ET_GazeRighty', 'ET_ValidityLeftEye', 'ET_ValidityRightEye', 'ET_GyroX', 'ET_GyroY', 'ET_GyroZ', 'ET_AccX', 'ET_AccY', 'ET_AccZ', 'ET_HeadRotationPitch', 'ET_HeadRotationYaw']

✅ PHASE 1A FINAL CLEANING PIPELINE COMPLETE
